# gpt2-cuda on Colab

1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells in order. Cell 2 asks you to upload `gpt2-cuda.zip` (or set `USE_GITHUB = True`).

**After Runtime → Restart session:** rerun the cell under "After a restart" below instead of re-uploading. The files and the built kernels survive a restart; the working folder and Python's memory don't.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Get the code. Upload the zip, or set USE_GITHUB = True if the repo has the latest code.
USE_GITHUB = False
REPO = "https://github.com/asmipulgam/gpt2-cuda.git"

%cd /content
!rm -rf gpt2-cuda
if USE_GITHUB:
    !git clone {REPO} gpt2-cuda
else:
    from google.colab import files
    name = next(iter(files.upload()))  # click "Choose Files", pick gpt2-cuda.zip
    !unzip -q -o "{name}" -d gpt2-cuda
%cd /content/gpt2-cuda
%env PYTHONPATH=/content/gpt2-cuda
!ls  # should list setup.py, csrc, gpt2_cuda, tests, benchmarks

In [ ]:
# 3. Install dependencies and compile the CUDA kernels (a few minutes).
# 7.5 = T4. Use 8.0 for an A100, 8.9 for an L4.
!pip install -q -r requirements.txt
!TORCH_CUDA_ARCH_LIST="7.5" python setup.py build_ext --inplace 2>&1 | tail -40
!ls gpt2_cuda/*.so  # must show a _C...so file; if not, the build failed above

### After a restart
Only run this after Runtime → Restart session (skip it on a fresh run).

In [ ]:
%cd /content/gpt2-cuda
%env PYTHONPATH=/content/gpt2-cuda

In [ ]:
# 4. Correctness tests. The summary line should be mostly "passed", not "skipped".
!python -m pytest -q tests

In [ ]:
# 5. Baseline: Hugging Face GPT-2 in PyTorch eager mode
!python benchmarks/baseline.py --batch-sizes 1 8 --prompt-lens 32 256

In [ ]:
# 6. Kernel and end-to-end benchmarks, then charts
!python benchmarks/bench_kernels.py
!python benchmarks/bench_e2e.py
!python benchmarks/plot.py

In [ ]:
# 7. Show the results
from IPython.display import Image, Markdown, display
from pathlib import Path

results = Path('benchmarks/results')
for md in ['baseline.md', 'kernels.md', 'e2e.md']:
    if (results / md).exists():
        display(Markdown((results / md).read_text()))
for png in sorted(results.glob('*.png')):
    display(Image(str(png)))

In [ ]:
# 8. Try it: generate text with the fused kernels
import torch
from transformers import GPT2TokenizerFast
from gpt2_cuda import GPT2

tok = GPT2TokenizerFast.from_pretrained('gpt2')
model = GPT2.from_pretrained('gpt2', backend='cuda_flash')
ids = tok('The future of GPU computing is', return_tensors='pt').input_ids.cuda()
print(tok.decode(model.generate(ids, 40)[0]))

In [ ]:
# 9. Nsight Compute. Colab and Kaggle block GPU performance counters (ERR_NVGPUCTRPERM);
# it needs a rented cloud GPU VM (root access). Optional: skip it here.
!bash benchmarks/ncu.sh

In [ ]:
# 10. Download the results before the session ends (Colab deletes everything)
!zip -qr results.zip benchmarks/results
from google.colab import files
files.download('results.zip')